In [1]:
import time
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPIError
%load_ext google.cloud.bigquery
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"]="C:/Users/yutin/OneDrive - The University of Texas at Austin/keys/auspicious-lead-382321-38c1a1631705.json"

# Entity Merge

In this notebook, we merge the `imdb_movie` records with the tconst values when their corresponding movie_ids are present in the `Tmdb_Movie` table. We output a new table, `movie_stg.Movie`, with the merged results. We use the bridge table `tmdb_external_ids` to merge them. 

In [9]:
%%bigquery
select * from movie_raw.imdb_movie
order by tconst desc
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,tconst,title,release_year,runtime_minutes,genres,directors,writers,average_rating,num_ratings,actors,data_source,load_time
0,tt10146070,Skin Deep: The Battle Over Morgellons,2019,90,Documentary,Pi Ware,Pi Ware,7.3,134,None,imdb,2024-02-10 01:48:02.810151+00:00
1,tt10145122,The Perfect Secret,2019,120,"Comedy,Drama,Romance",Bora Dagtekin,None,6.6,6875,"Valentin von Falkenhayn, Henri Wambach, Bruno ...",imdb,2024-02-10 01:48:02.810151+00:00
2,tt10144276,Y. Köyü Ye'cüc Me'cüc,2020,<NA>,Horror,Batuhan Gündüz,Batuhan Gündüz,7.2,64,"Onur Ertus, Batuhan Gündüz, Tugay Kurt, Burak ...",imdb,2024-02-10 01:48:02.810151+00:00
3,tt10144210,Keepsake!,2019,<NA>,Horror,Nathan Hine,Nathan Hine,6.2,11,"Nathan Hine, Tiffany Laskey, Bob Vresilovic",imdb,2024-02-10 01:48:02.810151+00:00
4,tt10143498,My Dog Stupid,2019,106,"Comedy,Drama",Yvan Attal,None,6.2,2329,"Stéphane Guillon, Louis Barge, Karlos Hébrard,...",imdb,2024-02-10 01:48:02.810151+00:00


In [8]:
%%bigquery
select * from movie_raw.tmdb_external_ids
order by imdb_id desc
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,tmdb_id,imdb_id,wikidata_id,facebook_id,instagram_id,twitter_id,data_source,load_time
0,630392,tt9907782,None,None,None,None,tmdb,2024-02-07 03:52:48.528082+00:00
1,1186957,tt9900816,Q123401570,None,None,None,tmdb,2024-02-07 03:52:48.528082+00:00
2,870671,tt9898844,None,None,None,None,tmdb,2024-02-07 03:52:48.528082+00:00
3,595172,tt9896916,None,PilgrimsProgressMovie,None,None,tmdb,2024-02-07 03:52:48.528082+00:00
4,920258,tt9896768,None,None,None,None,tmdb,2024-02-07 03:52:48.528082+00:00


In [5]:
%%bigquery
select * from movie_stg.Tmdb_Movie
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,movie_id,title,original_language,overview,popularity,release_date,genre_ids,vote_average,vote_count,load_time
0,1160025,Naga,ar,Title: Naga (2023). Stranded amid Riyadh's des...,33.943,9/8/2023,"[53, 35, 18]",5.3,19,2024-02-08 23:34:20.679895+00:00
1,1153366,Head to Head,ar,Title: Head to Head (2023). Hilarity meets dan...,20.481,8/3/2023,"[35, 53, 28]",6.3,59,2024-02-08 23:34:20.679895+00:00
2,1221388,From the Ashes,ar,Title: From the Ashes (2024). At a strict girl...,66.267,1/18/2024,"[53, 18]",6.3,15,2024-02-08 23:34:20.679895+00:00
3,320956,The Cuckold,ar,Title: The Cuckold (2009). The Cuckold is a co...,19.454,8/11/2009,[18],2.8,6,2024-02-08 23:34:20.679895+00:00
4,659676,Masameer: The Movie,ar,"Title: Masameer: The Movie (2020). Danah, a Sa...",46.115,1/9/2020,"[16, 35, 28, 18]",6.1,28,2024-02-08 23:34:20.679895+00:00


To eliminate needing the external_ids as a bridge table everytime, we create a table containing movies that're present in both tmdb and imdb and also have both a tmdb movie id and an imdb movie id. (3913/9923)

In [7]:
%%bigquery

CREATE or REPLACE TABLE movie_stg.tmdb_imdb AS (
    SELECT * FROM (
        SELECT 
            t.tmdb_id,
            t.imdb_id,
            tm.title AS tmdb_title,
            tm.original_language,
            tm.overview,
            tm.release_date,
            tm.genre_ids as tmdb_genre_ids,
            tm.data_source,
            tm.load_time as tmdb_load_time,
            im.title AS imdb_title,
            im.release_year AS imdb_release_year,
            im.runtime_minutes AS imdb_runtime_minutes,
            im.directors,
            im.writers,
            im.average_rating,
            im.num_ratings,
            im.actors AS imdb_actors,
            im.data_source AS imdb_data_source
          FROM 
            movie_raw.tmdb_external_ids t
          JOIN 
            movie_stg.Tmdb_Movie tm ON t.tmdb_id = tm.movie_id
          JOIN 
            movie_raw.imdb_movie im ON t.imdb_id = im.tconst
    )
);

Query is running:   0%|          |

""


In [11]:
%%bigquery
SELECT COUNT(*) from movie_stg.tmdb_imdb;

Query is running:   0%|          |

Downloading:   0%|          |

,f0_
0,3913


# runtime_minutes

Merge `tmdb_imdb.tconst` into `Tmdb_Movie.imdb_id`, to enrich the movie records with the runtime_minutes when it is missing from `Tmdb_Movie`. Only a small subset will be enriched (3910/9923).

In [14]:
%%bigquery
    SELECT t.*, ti.imdb_runtime_minutes AS runtime_minutes from movie_stg.Tmdb_Movie t
    JOIN movie_stg.tmdb_imdb ti
    ON ti.tmdb_id = t.movie_id
    where ti.imdb_runtime_minutes is not null
    limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,movie_id,title,original_language,overview,release_date,genre_ids,data_source,load_time,runtime_minutes
0,646749,Yu Pui Tsuen,cn,Adapted from the same infamous ancient text as...,1986-12-12,[],tmdb,2024-02-08 23:34:20.679895+00:00,91
1,220590,Emmanuelle in Hong Kong,cn,Yan and Emily have been married for two years....,2003-06-06,[18],tmdb,2024-02-08 23:34:20.679895+00:00,95
2,9060,Dumplings,cn,A rich woman is losing her attractiveness and ...,2004-08-19,[27],tmdb,2024-02-08 23:34:20.679895+00:00,91
3,68590,Legacy of Rage,cn,"Brandon plays Brandon Ma, a regular working Jo...",1986-12-20,[28],tmdb,2024-02-08 23:34:20.679895+00:00,90
4,10619,Once Upon a Time in China III,cn,Master Wong and his disciples enroll in the 'D...,1993-02-11,[28],tmdb,2024-02-08 23:34:20.679895+00:00,125


In [24]:
%%bigquery
  create or replace table movie_stg.Movie as
    select t.movie_id as tmdb_id, ti.imdb_id, t.* EXCEPT(movie_id, data_source, load_time), ti.imdb_runtime_minutes as duration, t.data_source, t.load_time
    from movie_stg.Tmdb_Movie t
    left join movie_stg.tmdb_imdb ti
    on ti.tmdb_id = t.movie_id;

Query is running:   0%|          |

""


In [25]:
%%bigquery
  select * from movie_stg.Movie
  where duration is not null
  limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,tmdb_id,imdb_id,title,original_language,overview,release_date,genre_ids,duration,data_source,load_time
0,79871,tt0107565,Crazy Love,cn,What was supposed to be summer school in Engla...,1993-04-17,"[35, 10749, 18]",77,tmdb,2024-02-08 23:34:20.679895+00:00
1,221912,tt0202543,Raped by an Angel 4: The Rapist's Union,cn,CIA agent turned HIV positive sexual predator ...,1999-01-17,"[53, 35, 27]",81,tmdb,2024-02-08 23:34:20.679895+00:00
2,13345,tt0112778,A Chinese Odyssey Part One: Pandora's Box,cn,When the Goddess of Happiness tosses the Longe...,1995-01-21,"[28, 12, 14, 35, 10749]",87,tmdb,2024-02-08 23:34:20.679895+00:00
3,192888,tt0104807,3 Days of a Blind Girl,cn,"The wife of a heart surgeon, suffering from te...",1993-05-27,[53],88,tmdb,2024-02-08 23:34:20.679895+00:00
4,105904,tt0311792,Naked Poison,cn,Nerdy office worker Min gets his kicks taking ...,2000-09-23,"[27, 53]",88,tmdb,2024-02-08 23:34:20.679895+00:00


The only problem is that the `data_source` should no longer be "bird" for the merged records. Update them to "bird_faker", to indicate that they came from both data sources:  

In [19]:
%%bigquery
update movie_stg.Movie set data_source = 'tmdb_imdb' where duration is not null;

Query is running:   0%|          |

""


In [20]:
%%bigquery
  select * from movie_stg.Movie
  where duration is not null
  limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,movie_id,title,original_language,overview,release_date,genre_ids,duration,data_source,load_time
0,10655,Gettysburg,en,"In the summer of 1863, General Robert E. Lee l...",1993-10-08,"[18, 36, 10752]",271,tmdb_imdb,2024-02-08 23:34:20.679895+00:00
1,39103,Dragon Ball Z: The Return of Cooler,ja,Cooler has resurrected himself as a robot and ...,1992-03-07,"[28, 16, 878]",45,tmdb_imdb,2024-02-08 23:34:20.679895+00:00
2,83088,Saint Seiya: Warriors of the Final Holy Battle,ja,The plot involves the coming of Lucifer to the...,1989-03-18,"[28, 16, 14]",45,tmdb_imdb,2024-02-08 23:34:20.679895+00:00
3,39145,Dragon Ball: Sleeping Princess in Devil's Castle,ja,Goku and Kuririn are given an assignment by Ka...,1987-07-18,"[28, 16, 14, 27]",45,tmdb_imdb,2024-02-08 23:34:20.679895+00:00
4,83091,Saint Seiya: The Heated Battle of the Gods,ja,"In Siberia, Hyōga saves a man that is being at...",1988-03-12,"[28, 16, 14]",45,tmdb_imdb,2024-02-08 23:34:20.679895+00:00


In [21]:
%%bigquery
  select count(*) duration_count
  from movie_stg.Movie
  where duration is not null

Query is running:   0%|          |

Downloading:   0%|          |

,duration_count
0,3910


In [23]:
%%bigquery
  select (select count(*) from movie_stg.Movie) as merged_table_count,
  (select count(*) from movie_stg.Tmdb_Movie) as tmdb_table_count

Query is running:   0%|          |

Downloading:   0%|          |

,merged_table_count,tmdb_table_count
0,9923,9923


# directors, writers

Find all the `Movie` records with a null directors and writers whose directors and writers exists in `imdb_movie`. Merge `imdb_movie.directors` and `imdb_movie.writers` into `Movie.directors` and `Movie.writers`to enrich those records. Only a small subset of the records will be enriched (directors: 3592/9923; writers: 951/9923).

In [28]:
%%bigquery
select m.*, ti.directors
from movie_stg.tmdb_imdb ti
join movie_stg.Movie m
on ti.imdb_id = m.imdb_id
where ti.directors is not null
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,tmdb_id,imdb_id,title,original_language,overview,release_date,genre_ids,duration,data_source,load_time,directors
0,580,tt0093300,Jaws: The Revenge,en,"After another deadly shark attack, Ellen Brody...",1987-07-17,"[12, 53, 27]",89,tmdb,2024-02-08 23:34:20.679895+00:00,Joseph Sargent
1,8854,tt0120207,Steel,en,When a renegade military reject puts new super...,1997-08-15,"[878, 28, 12]",97,tmdb,2024-02-08 23:34:20.679895+00:00,Kenneth Johnson
2,16456,tt0353324,The Crow: Wicked Prayer,en,When ex-convict Jimmy and his girlfriend are b...,2005-05-04,"[28, 27, 14]",99,tmdb,2024-02-08 23:34:20.679895+00:00,Lance Mungia
3,11546,tt0110857,Police Academy: Mission to Moscow,en,The Russians seek help in dealing with the Maf...,1994-06-09,"[35, 80]",83,tmdb,2024-02-08 23:34:20.679895+00:00,Alan Metter
4,56386,tt0074768,SS Experiment Love Camp,it,"Near the end of WW2, prisoners of war are used...",1976-11-16,"[27, 10752]",94,tmdb,2024-02-08 23:34:20.679895+00:00,Sergio Garrone


In [32]:
%%bigquery
select m.*, ti.writers
from movie_stg.tmdb_imdb ti
join movie_stg.Movie m
on ti.imdb_id = m.imdb_id
where ti.writers is not null
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,tmdb_id,imdb_id,title,original_language,overview,release_date,genre_ids,duration,directors,writers,data_source,load_time,writers_1
0,683080,tt0101792,El día que nací yo,es,"A gypsy who sells fish in a market, sings on w...",1991-09-25,"[18, 10749, 10402]",<NA>,Pedro Olea,Jaime de Armiñán,tmdb,2024-02-08 23:34:20.679895+00:00,Jaime de Armiñán
1,38142,tt0983213,5 Centimeters per Second,ja,Three moments in Takaki's life: his relationsh...,2007-03-03,"[16, 18, 10751, 10749]",63,Makoto Shinkai,Makoto Shinkai,tmdb,2024-02-08 23:34:20.679895+00:00,Makoto Shinkai
2,41428,tt0096251,Tetsuo: The Iron Man,ja,"A ""metal fetishist"", driven mad by the maggots...",1989-07-01,"[27, 878]",67,Shin'ya Tsukamoto,Shin'ya Tsukamoto,tmdb,2024-02-08 23:34:20.679895+00:00,Shin'ya Tsukamoto
3,39858,tt0224102,Entrails of a Beautiful Woman,ja,A nurse investigates a sex trafficking ring af...,1986-09-23,"[27, 878]",68,Kazuo 'Gaira' Komizu,Kazuo 'Gaira' Komizu,tmdb,2024-02-08 23:34:20.679895+00:00,Kazuo 'Gaira' Komizu
4,13666,tt0034398,The Wolf Man,en,"After his brother's death, Larry Talbot return...",1941-12-12,"[27, 18]",70,George Waggner,Curt Siodmak,tmdb,2024-02-08 23:34:20.679895+00:00,Curt Siodmak


In [31]:
%%bigquery
  create or replace table movie_stg.Movie as
    select m.tmdb_id, m.imdb_id, m.* EXCEPT(tmdb_id, imdb_id, data_source, load_time), ti.directors as directors, ti.writers as writers, m.data_source, m.load_time
    from movie_stg.Movie m
    left join movie_stg.tmdb_imdb ti
    on ti.tmdb_id = m.tmdb_id;

Query is running:   0%|          |

""


In [35]:
%%bigquery
  select * from movie_stg.Movie
  where directors IS NOT NULL LIMIT 5

Query is running:   0%|          |

Downloading:   0%|          |

,tmdb_id,imdb_id,title,original_language,overview,release_date,genre_ids,duration,directors,writers,data_source,load_time
0,683080,tt0101792,El día que nací yo,es,"A gypsy who sells fish in a market, sings on w...",1991-09-25,"[18, 10749, 10402]",<NA>,Pedro Olea,Jaime de Armiñán,tmdb,2024-02-08 23:34:20.679895+00:00
1,10655,tt0107007,Gettysburg,en,"In the summer of 1863, General Robert E. Lee l...",1993-10-08,"[18, 36, 10752]",271,Ron Maxwell,None,tmdb,2024-02-08 23:34:20.679895+00:00
2,39103,tt0142237,Dragon Ball Z: The Return of Cooler,ja,Cooler has resurrected himself as a robot and ...,1992-03-07,"[28, 16, 878]",45,Daisuke Nishio,None,tmdb,2024-02-08 23:34:20.679895+00:00
3,39145,tt0142249,Dragon Ball: Sleeping Princess in Devil's Castle,ja,Goku and Kuririn are given an assignment by Ka...,1987-07-18,"[28, 16, 14, 27]",45,Daisuke Nishio,None,tmdb,2024-02-08 23:34:20.679895+00:00
4,83091,tt0322918,Saint Seiya: The Heated Battle of the Gods,ja,"In Siberia, Hyōga saves a man that is being at...",1988-03-12,"[28, 16, 14]",45,Shigeyasu Yamauchi,None,tmdb,2024-02-08 23:34:20.679895+00:00


In [37]:
%%bigquery
  select count(*) directors_count
  from movie_stg.Movie
  where directors is not null

Query is running:   0%|          |

Downloading:   0%|          |

,directors_count
0,3592


In [38]:
%%bigquery
  select count(*) writers_count
  from movie_stg.Movie
  where writers is not null

Query is running:   0%|          |

Downloading:   0%|          |

,writers_count
0,951


In [43]:
%%bigquery
update movie_stg.Movie set data_source = 'tmdb_imdb' where directors is not null or writers is not null;

Query is running:   0%|          |

""


In [50]:
%%bigquery
  select *
  from movie_stg.Movie
  where writers is not null  and directors is null limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,tmdb_id,imdb_id,title,original_language,overview,release_date,genre_ids,duration,directors,writers,data_source,load_time
0,9023,tt0166813,Spirit: Stallion of the Cimarron,en,As a wild stallion travels across the frontier...,2002-05-24,"[16, 12, 35, 10751, 18, 37]",83,None,John Fusco,tmdb_imdb,2024-02-08 23:34:20.679895+00:00
1,41280,tt0082948,Private Lessons,en,"Phillip Filmore is a naive, 15-year-old, preoc...",1981-08-21,"[35, 10749]",87,None,Dan Greenburg,tmdb_imdb,2024-02-08 23:34:20.679895+00:00
2,14411,tt0165982,Sinbad: Legend of the Seven Seas,en,The sailor of legend is framed by the goddess ...,2003-07-02,"[10751, 16, 12, 35, 14, 28]",88,None,John Logan,tmdb_imdb,2024-02-08 23:34:20.679895+00:00
3,14369,tt0253798,Out Cold,en,Rick Rambis and his friends are having the tim...,2001-11-21,[35],89,None,Jon Zack,tmdb_imdb,2024-02-08 23:34:20.679895+00:00
4,25087,tt0112536,Bloodsport II,en,After thief Alex Cardo gets caught while steal...,1996-03-01,"[28, 53]",90,None,Jeff Schechter,tmdb_imdb,2024-02-08 23:34:20.679895+00:00


# Primary Key

In [39]:
%%bigquery
alter table movie_stg.Movie
  add primary key (tmdb_id) not enforced;

Query is running:   0%|          |

""


In [40]:
%%bigquery
select tmdb_id, count(*) duplicate_records
from movie_stg.Movie
group by tmdb_id
having count(*) > 1
order by count(*) desc

Query is running:   0%|          |

Downloading: |          |

,tmdb_id,duplicate_records


Creating a single attribute for the PK of `actor` and `character`

In [4]:
%%bigquery
create or replace table movie_stg.character as
    SELECT concat(person_id,'-',movie_id, '-',credit_order) as character_id, *
    FROM movie_raw.tmdb_character

Query is running:   0%|          |

""


In [5]:
%%bigquery
create or replace table movie_stg.actor as
    SELECT concat(person_id,'-',movie_id, '-',credit_order) as character_id, *
    FROM movie_raw.tmdb_actor

Query is running:   0%|          |

""


In [8]:
%%bigquery
select COUNT(distinct character_id) as distinct_count, COUNT(character_id) as raw_count from movie_stg.character

Query is running:   0%|          |

Downloading:   0%|          |

,distinct_count,raw_count
0,317256,317256


In [9]:
%%bigquery
select COUNT(distinct character_id) as distinct_count, COUNT(character_id) as raw_count from movie_stg.actor

Query is running:   0%|          |

Downloading:   0%|          |

,distinct_count,raw_count
0,322632,322632


# Foreign Keys will be addressed in catchall.ipynb

# Cleanup

In [41]:
%%bigquery
drop table movie_stg.tmdb_imdb

Query is running:   0%|          |

""


In [42]:
%%bigquery
drop table movie_stg.Tmdb_Movie

Query is running:   0%|          |

""
